<a href="https://colab.research.google.com/github/todd-jang/simulation_project/blob/main/Gotham_Deployment_Package.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import os
import torch
import numpy as np
from ray.rllib.algorithms.ppo import PPO

class GothamStandaloneController:
    """
    고담 시스템 배포용 Standalone 컨트롤러
    학습된 배트맨(휴머노이드)과 스파이더 카(자율주행) 모델을 로드하여 실시간 제어 수행
    """
    def __init__(self, model_path_car, model_path_batman):
        # 1. 경량화된 추론 엔진 로드 (실제 배포시에는 ONNX Runtime이나 TensorRT 사용 권장)
        self.car_policy = self._load_model(model_path_car)
        self.batman_policy = self._load_model(model_path_batman)
        print("🦇 [Gotham Hub] Standalone Controller Initialized.")

    def _load_model(self, path):
        # 학습된 체크포인트에서 가중치만 추출하여 로드
        return torch.jit.load(path) if path.endswith('.pt') else path

    def get_action(self, agent_id, observation):
        """실시간 센서 데이터를 받아 제어 명령(Action) 출력"""
        obs_tensor = torch.from_numpy(observation).float()
        with torch.no_grad():
            if agent_id == "spider_car":
                action = self.car_policy(obs_tensor)
            else:
                action = self.batman_policy(obs_tensor)
        return action.numpy()

# ─── 배포용 패키지 빌드 스크립트 (CLI) ───

def export_to_onnx(algorithm, export_dir):
    """Ray RLlib 모델을 Standalone 배포용 ONNX 포맷으로 변환"""
    os.makedirs(export_dir, exist_ok=True)

    # 스파이더 카 정책 추출
    algorithm.export_policy_model(
        export_dir + "/spider_car_onnx",
        policy_id="spider_car_policy"
    )

    # 배트맨 휴머노이드 정책 추출
    algorithm.export_policy_model(
        export_dir + "/batman_policy_onnx",
        policy_id="batman_humanoid_policy"
    )
    print(f"📦 [Build Success] Standalone packages exported to: {export_dir}")

def create_dockerfile():
    """배포를 위한 Dockerfile 생성 스크립트"""
    docker_content = """
FROM python:3.9-slim
WORKDIR /app
COPY . /app
RUN pip install torch numpy onnxruntime
CMD ["python", "gotham_controller_run.py"]
    """
    with open("Dockerfile", "w") as f:
        f.write(docker_content)
    print("🐳 Dockerfile generated for standalone deployment.")


In [9]:
# Mock a trained algorithm object for demonstration purposes
class MockAlgorithm:
    def export_policy_model(self, export_path, policy_id):
        print(f"Mock exporting policy '{policy_id}' to '{export_path}'")
        # Simulate creating dummy directories/files
        import os
        os.makedirs(export_path, exist_ok=True)
        with open(os.path.join(export_path, f"{policy_id}.mock"), "w") as f:
            f.write(f"Mock model data for {policy_id}")
        print(f"Created mock file: {os.path.join(export_path, f'{policy_id}.mock')}")

# Instantiate the mock algorithm
my_trained_algorithm = MockAlgorithm()

# Define an export directory for the demo
export_dir = "./dist/gotham_v1_demo"

# Call the export_to_onnx function with the mock algorithm
export_to_onnx(my_trained_algorithm, export_dir)

# Call the create_dockerfile function
create_dockerfile()

Mock exporting policy 'spider_car_policy' to './dist/gotham_v1_demo/spider_car_onnx'
Created mock file: ./dist/gotham_v1_demo/spider_car_onnx/spider_car_policy.mock
Mock exporting policy 'batman_humanoid_policy' to './dist/gotham_v1_demo/batman_policy_onnx'
Created mock file: ./dist/gotham_v1_demo/batman_policy_onnx/batman_humanoid_policy.mock
📦 [Build Success] Standalone packages exported to: ./dist/gotham_v1_demo
🐳 Dockerfile generated for standalone deployment.


In [10]:
import os

print(f"Contents of '{export_dir}':")
if os.path.exists(export_dir):
    for root, dirs, files in os.walk(export_dir):
        level = root.replace(export_dir, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            print(f'{subindent}{f}')
else:
    print("Export directory does not exist. Please ensure the previous cell ran successfully.")

print("\nAlso checking for Dockerfile in current directory:")
if os.path.exists("Dockerfile"):
    with open("Dockerfile", "r") as f:
        print(f.read())
else:
    print("Dockerfile not found.")

Contents of './dist/gotham_v1_demo':
gotham_v1_demo/
    spider_car_onnx/
        spider_car_policy.mock
    batman_policy_onnx/
        batman_humanoid_policy.mock

Also checking for Dockerfile in current directory:

FROM python:3.9-slim
WORKDIR /app
COPY . /app
RUN pip install torch numpy onnxruntime
CMD ["python", "gotham_controller_run.py"]
    


In [11]:
import inspect

print("--- Source code for export_to_onnx ---")
print(inspect.getsource(export_to_onnx))
print("\n--- Source code for create_dockerfile ---")
print(inspect.getsource(create_dockerfile))

--- Source code for export_to_onnx ---
def export_to_onnx(algorithm, export_dir):
    """Ray RLlib 모델을 Standalone 배포용 ONNX 포맷으로 변환"""
    os.makedirs(export_dir, exist_ok=True)

    # 스파이더 카 정책 추출
    algorithm.export_policy_model(
        export_dir + "/spider_car_onnx",
        policy_id="spider_car_policy"
    )

    # 배트맨 휴머노이드 정책 추출
    algorithm.export_policy_model(
        export_dir + "/batman_policy_onnx",
        policy_id="batman_humanoid_policy"
    )
    print(f"📦 [Build Success] Standalone packages exported to: {export_dir}")


--- Source code for create_dockerfile ---
def create_dockerfile():
    """배포를 위한 Dockerfile 생성 스크립트"""
    docker_content = """
FROM python:3.9-slim
WORKDIR /app
COPY . /app
RUN pip install torch numpy onnxruntime
CMD ["python", "gotham_controller_run.py"]
    """
    with open("Dockerfile", "w") as f:
        f.write(docker_content)
    print("🐳 Dockerfile generated for standalone deployment.")



In [12]:
pip install 'ray[rllib]'